<a href="https://colab.research.google.com/github/miguel-angel-03/Market-Sizing-Wellness-Analytics/blob/main/Market_Sizing_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Feasibility Study: Data-Driven Market Sizing & Financial Modeling**

**Executive Summary**

**Project Overview:** This project executes a comprehensive market sizing and viability study for a conceptual Colombian-based activewear (*athleisure*) and wellness tech ecosystem targeting consumers aged 18 to 35. By leveraging macroeconomic microdata, digital adoption metrics, and industry-specific indicators, this analysis quantifies the financial potential and operational feasibility of a subscription-based business model for the 2025 fiscal year.

**Business Objective:** The primary goal is to transition from raw national population data to a high-precision SOM (Serviceable Obtainable Market), mapping its precise geographic concentration and demographic distribution to validate a recurring revenue model.

**Methodology (CRISP-DM Framework):** This project strictly adheres to the Cross-Industry Standard Process for Data Mining (CRISP-DM), ensuring a robust, enterprise-grade analytical pipeline:
1. **Business Understanding:** Defining the Target SOM, pricing strategy ($35k COP ticket), and projecting Monthly/Annual Recurring Revenue (MRR/ARR) for the subscription model.
2. **Data Understanding:** Profiling and diagnosing the DANE GEIH macro-economic microdata to identify structural demographics, geographic identifiers, and formal labor income variables.
3. **Data Preparation (Phase I & II):** Executing an automated ETL pipeline to ingest, clean, map, and expand nearly 1 million raw survey records into a centralized Google BigQuery Data Warehouse.
4. **Modeling (Phase II & III):** Applying strategic funnel constraints (minimum income >= 1.5M COP, 40% wellness engagement, 79% digital banking readiness) to calculate TAM, SAM, and SOM.
5. **Evaluation (Phase III):** Validating the operational viability of the SOM through geographic clustering (applying a 5% relevance threshold) and gender distribution analysis.
6. **Deployment (Phase IV):** Exporting the finalized, optimized data models to an enterprise BI resource layer for interactive, multi-dimensional consumption in Power BI.

## **Phase I: Data Engineering & ETL Pipeline**

**The Challenge:** Processing longitudinal government microdata presents several architectural challenges, including dataset fragmentation (48 separate monthly CSV files nested in 12 directories), schema drift (unannounced modifications to variable nomenclature between annual series), and encoding anomalies inherent to Latin American character sets.

To ensure data integrity and optimize cloud computing resources, the pipeline is divided into a two-stage Data Engineering process:
1. **Stage 1:** Zero-Memory Schema Profiling (Diagnostic).
2. **Stage 2:** Production Ingestion via Static Schema Mapping to a Google BigQuery Data Warehouse.

### Stage 1: Zero-Memory Schema Profiling and Diagnostics

**Objective:** Before enforcing a static schema mapping in a production ETL pipeline, it is imperative to execute a diagnostic extraction of the source schema to mitigate pipeline failures caused by schema drift.

**Methodology:** This diagnostic script operates as an automated Data Profiler. It systematically traverses the cloud storage root directory to isolate one representative sample file for each targeted data domain (Demographics, Employment, Housing, and Other Income). It executes a zero-memory load (`nrows=0` in Pandas), extracting strictly the header array without ingesting the payload, applying `ISO-8859-1` (Latin-1) decoding to bypass codec exceptions.

In [1]:
# ==============================================================================
# STAGE 1: DIAGNOSTIC SCRIPT: ZERO-MEMORY SCHEMA PROFILING
# ==============================================================================

import os
import pandas as pd
import unicodedata
from google.colab import auth
from google.colab import drive

# 1. Environment Setup
print("Initializing environment for Data Profiling...")
auth.authenticate_user()
drive.mount('/content/drive')

ROOT_DIRECTORY = '/content/drive/MyDrive/GEIH/'

# The 4 file categories we need to profile based on filename patterns
TARGET_CATEGORIES = ['caracteristicas', 'ocupados', 'vivienda', 'otros']

def normalize_text(text):
    """Removes accents to ensure accurate pattern matching."""
    return unicodedata.normalize('NFKD', str(text)).encode('ASCII', 'ignore').decode('utf-8').lower()

# 2. Directory Scan and Sample Identification
print("\nScanning directory for sample files...")
sample_files = {}

for root, dirs, files in os.walk(ROOT_DIRECTORY):
    for file in files:
        if file.lower().endswith('.csv'):
            normalized_name = normalize_text(file)

            # Find one sample for each category
            for category in TARGET_CATEGORIES:
                if category in normalized_name and category not in sample_files:
                    sample_files[category] = os.path.join(root, file)

            # Stop searching if we found all 4 samples
            if len(sample_files) == 4:
                break
    if len(sample_files) == 4:
        break

# 3. Header Extraction and Output
print("\n================ SCHEMA EXTRACT ================\n")

for category, file_path in sample_files.items():
    filename = os.path.basename(file_path)
    try:
        # Extract only the header row using Latin-1 encoding
        header_df = pd.read_csv(file_path, sep=';', nrows=0, encoding='latin-1')
        columns_list = header_df.columns.tolist()

        print(f"CATEGORY: {category.upper()}")
        print(f"SOURCE FILE: {filename}")
        print(f"COLUMNS ARRAY:\n{columns_list}\n")
        print("-" * 60 + "\n")

    except Exception as error:
        print(f"Error profiling {filename}: {error}\n")

print("Profiling Complete. Awaiting manual schema mapping.")

Initializing environment for Data Profiling...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Scanning directory for sample files...

================ SCHEMA EXTRACT ================

CATEGORY: VIVIENDA
SOURCE FILE: Datos del hogar y la vivienda.CSV
COLUMNS ARRAY:
['PERIODO', 'MES', 'PER', 'DIRECTORIO', 'SECUENCIA_P', 'HOGAR', 'REGIS', 'AREA', 'CLASE', 'FEX_C18', 'DPTO', 'P4005', 'P4010', 'P4020', 'P4030S1', 'P4030S1A1', 'P4030S2', 'P4030S3', 'P4030S4', 'P4030S4A1', 'P4030S5', 'P70', 'P5000', 'P5010', 'P5020', 'P5030', 'P5040', 'P5050', 'P5070', 'P5080', 'P5090', 'P5090S1', 'P5100', 'P5110', 'P5130', 'P5140', 'P5222S1', 'P5222S2', 'P5222S3', 'P5222S4', 'P5222S5', 'P5222S6', 'P5222S7', 'P5222S8', 'P5222S8A1', 'P5222S9', 'P5222S10', 'P5222S11', 'P6008']

------------------------------------------------------------

CATEGORY: CARACTERISTICAS
SOURCE FILE: Características generales, seguridad social en salud

### Stage 2: Production Ingestion via Static Schema Mapping

**Objective:** To execute the permanent migration of the segmented macroeconomic time series into a centralized Google BigQuery Data Warehouse, optimizing the architecture for downstream SQL analytics.

**Methodology:** This stage functions as the primary ETL engine utilizing a Static Declarative Schema. By injecting the validated source arrays directly into the pipeline's memory, the system bypasses dynamic header resolution. This declarative approach eliminates exceptions associated with casing inconsistencies and nomenclature drift (e.g., standardizing the unannounced change to `INGLABO` in the 2025 series).

Crucially, this explicit schema extraction guarantees the inclusion of the `FEX_C18` (Expansion Factor) variable, which is strictly required to transition from raw survey sample counts to real-world national population projections. The pipeline enforces string data typing for composite primary keys to guarantee referential integrity before executing the persistent load to the cloud infrastructure.

In [2]:
# ==============================================================================
# STAGE 2: PRODUCTION INGESTION VIA STATIC SCHEMA MAPPING
# ==============================================================================
# Note: Execution assumes environment initialization, library imports,
# and Google Drive mounting were successfully completed in Stage 1.

import pandas as pd
import pandas_gbq
import os
import unicodedata

print("Initiating Stage 2: Declarative Schema Ingestion...")

PROJECT_ID = 'avani-market-research'
DATASET_ID = 'GEIH'
ROOT_DIRECTORY = '/content/drive/MyDrive/GEIH/'

# ------------------------------------------------------------------------------
# 1. STATIC SCHEMA DECLARATION (PROFILED ARRAYS)
# ------------------------------------------------------------------------------
PROFILED_HEADERS = {
    'caracteristicas': ['PERIODO', 'MES', 'PER', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR', 'REGIS', 'AREA', 'CLASE', 'FEX_C18', 'DPTO', 'PT', 'P6016', 'P3271', 'P6040', 'P6030S1', 'P6030S3', 'P6050', 'P6083', 'P6083S1', 'P6081', 'P6081S1', 'P2057', 'P2059', 'P2061', 'P6080', 'P6080S1', 'P6080S1A1', 'P6070', 'P6071', 'P6071S1', 'P6090', 'P6100', 'P6110', 'P6120', 'P1906S1', 'P1906S2', 'P1906S3', 'P1906S4', 'P1906S5', 'P1906S6', 'P1906S7', 'P1906S8', 'P6160', 'P6170', 'P3041', 'P3042', 'P3042S1', 'P3042S2', 'P3043', 'P3043S1', 'P3038', 'P3039', 'POB_MAY18'],
    'ocupados': ['PERIODO', 'MES', 'PER', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR', 'REGIS', 'AREA', 'CLASE', 'FEX_C18', 'DPTO', 'FT', 'P3044S2', 'P6440', 'P6450', 'P6460', 'P6460S1', 'P6400', 'P6410', 'P6422', 'P6420S2', 'P6424S1', 'P6424S2', 'P6424S3', 'P6424S5', 'P6426', 'P6430', 'P6430S1', 'P3045S1', 'P3045S2', 'P3045S3', 'P3046', 'P3363', 'P9440', 'P6500', 'P3364', 'P3364S1', 'P6510', 'P6510S1', 'P6510S2', 'P6590', 'P6590S1', 'P6600', 'P6600S1', 'P6610', 'P6610S1', 'P6620', 'P6620S1', 'P6585S1', 'P6585S1A1', 'P6585S1A2', 'P6585S2', 'P6585S2A1', 'P6585S2A2', 'P6585S3', 'P6585S3A1', 'P6585S3A2', 'P6585S4', 'P6585S4A1', 'P6585S4A2', 'P6545', 'P6545S1', 'P6545S2', 'P6580', 'P6580S1', 'P6580S2', 'P6630S1', 'P6630S1A1', 'P6630S2', 'P6630S2A1', 'P6630S3', 'P6630S3A1', 'P6630S4', 'P6630S4A1', 'P6630S6', 'P6630S6A1', 'P6640', 'P6640S1', 'P1800', 'P1800S1', 'P1801S1', 'P1801S2', 'P1801S3', 'P1802', 'P3047', 'P3048', 'P3049', 'P6765', 'P6765S1', 'P3051', 'P3052', 'P3053', 'P3365', 'P3365S1', 'P3054', 'P3054S1', 'P3055', 'P3055S1', 'P3056', 'P3057', 'P6760', 'P3058S1', 'P3058S2', 'P3058S3', 'P3058S4', 'P3058S5', 'P3059', 'P3061', 'P3062S1', 'P3062S2', 'P3062S3', 'P3062S4', 'P3062S5', 'P3062S6', 'P3062S7', 'P3062S8', 'P3062S9', 'P3063', 'P3063S1', 'P3064', 'P3064S1', 'P3065', 'P3066', 'P3067', 'P3067S1', 'P3067S2', 'P6775', 'P3068', 'P6750', 'P3073', 'P550', 'P6780', 'P6780S1', 'P1879', 'P1805', 'P6790', 'P6800', 'P6810', 'P6810S1', 'P6850', 'P6830', 'P6830S1', 'P3069', 'P6880', 'P6880S1', 'P6915', 'P6915S1', 'P6920', 'P6930', 'P6940', 'P6960', 'P6990', 'P9450', 'P7020', 'P760', 'P7026', 'P7028', 'P7028S1', 'P1880', 'P1880S1', 'P7040', 'P7045', 'P3071S3', 'P3072S2', 'P7050', 'P7070', 'P7075', 'P7077', 'P7090', 'P7100', 'P7110', 'P7120', 'P7130', 'P7140S1', 'P7140S2', 'P7140S3', 'P7140S4', 'P7140S5', 'P7140S6', 'P7140S7', 'P7140S8', 'P7140S9', 'P7140S9A1', 'P7150', 'P7160', 'P7170S1', 'P7170S5', 'P7170S6', 'P7180', 'P514', 'P515', 'P1881', 'P1881S1', 'P1882', 'P7240', 'P7240S1', 'OCI', 'INGLABO', 'RAMA2D_R4', 'RAMA4D_R4', 'OFICIO_C8'],
    'vivienda': ['PERIODO', 'MES', 'PER', 'DIRECTORIO', 'SECUENCIA_P', 'HOGAR', 'REGIS', 'AREA', 'CLASE', 'FEX_C18', 'DPTO', 'P4005', 'P4010', 'P4020', 'P4030S1', 'P4030S1A1', 'P4030S2', 'P4030S3', 'P4030S4', 'P4030S4A1', 'P4030S5', 'P70', 'P5000', 'P5010', 'P5020', 'P5030', 'P5040', 'P5050', 'P5070', 'P5080', 'P5090', 'P5090S1', 'P5100', 'P5110', 'P5130', 'P5140', 'P5222S1', 'P5222S2', 'P5222S3', 'P5222S4', 'P5222S5', 'P5222S6', 'P5222S7', 'P5222S8', 'P5222S8A1', 'P5222S9', 'P5222S10', 'P5222S11', 'P6008'],
    'otros': ['PERIODO', 'MES', 'PER', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR', 'REGIS', 'AREA', 'CLASE', 'FEX_C18', 'DPTO', 'P7495', 'P7500S1', 'P7500S1A1', 'P7500S2', 'P7500S2A1', 'P7500S3', 'P7500S3A1', 'P7505', 'P7510S1', 'P7510S1A1', 'P7510S2', 'P7510S2A1', 'P7510S3', 'P7510S3A1', 'P750S1', 'P750S1A1', 'P750S2', 'P750S2A1', 'P1661S5', 'P1661S5A1', 'P1661S6', 'P1661S6A1', 'P1661S3', 'P1661S3A1', 'P1661S4', 'P1661S4A1', 'P1661S4A2', 'P750S3', 'P750S3A1', 'P7510S5', 'P7510S5A1', 'P7510S6', 'P7510S6A1', 'P7510S7', 'P7510S7A1', 'P3367', 'P3368', 'P3369', 'P3370', 'P3370S1', 'P3371', 'P3371S1', 'P3371S2', 'P3371S3', 'P3371S4', 'P3372', 'P3372S1']
}

# The definitive business features required for the Data Warehouse schema.
# Note: INGLABO updated based on profiling. FEX_C18 ADDED for population expansion.
TARGET_FEATURES = {
    'caracteristicas': ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P6040', 'P6020', 'P3', 'FEX_C18', 'DPTO', 'P3271'],
    'ocupados': ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'INGLABO'],
    'vivienda': ['DIRECTORIO', 'SECUENCIA_P'],
    'otros': ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN']
}

# ------------------------------------------------------------------------------
# 2. FEATURE MAPPING FUNCTION
# ------------------------------------------------------------------------------
def map_exact_columns(category_key):
    """Maps the required uppercase features to the exact casing used in the source file."""
    source_array = PROFILED_HEADERS[category_key]

    if not source_array:
        return []

    casing_map = {col.upper(): col for col in source_array}
    required_upper = TARGET_FEATURES[category_key]

    return [casing_map[feat] for feat in required_upper if feat in casing_map]

# ------------------------------------------------------------------------------
# 3. DIRECTORY TRAVERSAL AND EXTRACTION
# ------------------------------------------------------------------------------
data_collections = {key: [] for key in PROFILED_HEADERS.keys()}
raw_file_registry = []

for root, dirs, files in os.walk(ROOT_DIRECTORY):
    for file in files:
        if file.lower().endswith('.csv'):
            raw_file_registry.append(os.path.join(root, file))

for current_file in raw_file_registry:
    filename_normalized = unicodedata.normalize('NFKD', str(os.path.basename(current_file))).encode('ASCII', 'ignore').decode('utf-8').lower()

    for category in PROFILED_HEADERS.keys():
        if category in filename_normalized:
            extracted_cols = map_exact_columns(category)

            if not extracted_cols:
                print(f"Warning: Missing required schema features for {os.path.basename(current_file)}. Skipping.")
                break

            try:
                casing_map = {col.upper(): col for col in PROFILED_HEADERS[category]}
                dtypes = {casing_map[pk]: str for pk in ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'] if pk in casing_map}

                temp_df = pd.read_csv(
                    current_file,
                    usecols=extracted_cols,
                    sep=';',
                    dtype=dtypes,
                    encoding='latin-1',
                    low_memory=False
                )

                temp_df.columns = [col.upper() for col in temp_df.columns]
                data_collections[category].append(temp_df)
                print(f"Ingested: {os.path.basename(current_file)} -> {category}")

            except Exception as error:
                print(f"Extraction Error in {os.path.basename(current_file)}: {error}")
            break

# ------------------------------------------------------------------------------
# 4. BIGQUERY MIGRATION
# ------------------------------------------------------------------------------
print("\nInitiating data migration to Google BigQuery...")

BQ_TABLES = {
    'caracteristicas': f'{DATASET_ID}.stg_caracteristicas_generales',
    'ocupados': f'{DATASET_ID}.stg_ocupados',
    'vivienda': f'{DATASET_ID}.stg_vivienda_hogares',
    'otros': f'{DATASET_ID}.stg_otros_inc'
}

for category, list_of_dfs in data_collections.items():
    if len(list_of_dfs) > 0:
        master_dataframe = pd.concat(list_of_dfs, ignore_index=True)
        destination_endpoint = BQ_TABLES[category]

        print(f"Executing Load: {len(master_dataframe)} rows into {destination_endpoint}...")

        pandas_gbq.to_gbq(
            master_dataframe,
            destination_endpoint,
            project_id=PROJECT_ID,
            if_exists='replace'
        )
        print(f"Integration Status: {destination_endpoint} successfully synchronized.")
    else:
        print(f"Log: No records found for category '{category}' or schema array missing.")

print("\nETL Stage 2 Finalized. Data Warehouse is now ready for SQL modeling.")

Initiating Stage 2: Declarative Schema Ingestion...
Ingested: Datos del hogar y la vivienda.CSV -> vivienda
Ingested: Características generales, seguridad social en salud y educación.CSV -> caracteristicas
Ingested: Ocupados.CSV -> ocupados
Ingested: Otros ingresos e impuestos.CSV -> otros
Ingested: Datos del hogar y la vivienda.CSV -> vivienda
Ingested: Características generales, seguridad social en salud y educación.CSV -> caracteristicas
Ingested: Ocupados.CSV -> ocupados
Ingested: Otros ingresos e impuestos.CSV -> otros
Ingested: Ocupados.CSV -> ocupados
Ingested: Datos del hogar y la vivienda.CSV -> vivienda
Ingested: Características generales, seguridad social en salud y educación.CSV -> caracteristicas
Ingested: Otros ingresos e impuestos.CSV -> otros
Ingested: Datos del hogar y la vivienda.CSV -> vivienda
Ingested: Características generales, seguridad social en salud y educación.CSV -> caracteristicas
Ingested: Otros ingresos e impuestos.CSV -> otros
Ingested: Ocupados.

100%|██████████| 1/1 [00:00<00:00, 7767.23it/s]


Integration Status: GEIH.stg_caracteristicas_generales successfully synchronized.
Executing Load: 358029 rows into GEIH.stg_ocupados...


100%|██████████| 1/1 [00:00<00:00, 10922.67it/s]


Integration Status: GEIH.stg_ocupados successfully synchronized.
Executing Load: 292942 rows into GEIH.stg_vivienda_hogares...


100%|██████████| 1/1 [00:00<00:00, 7695.97it/s]


Integration Status: GEIH.stg_vivienda_hogares successfully synchronized.
Executing Load: 653839 rows into GEIH.stg_otros_inc...


100%|██████████| 1/1 [00:00<00:00, 8830.11it/s]

Integration Status: GEIH.stg_otros_inc successfully synchronized.

ETL Stage 2 Finalized. Data Warehouse is now ready for SQL modeling.


## **Phase II: Data Modeling & Market Sizing (SQL-Python Integration)**

**Methodology:**
To execute the market sizing analysis, this project utilizes the **BigQuery Magics** for Google Colab. This allows for the direct execution of Standard SQL queries against the Data Warehouse while maintaining the results within the Python execution environment.

The integration ensures that macroeconomic data remains in the cloud for high-performance processing, while the resulting analytical aggregates are retrieved as Python DataFrames for final reporting and business visualization.

### 1. Total Addressable Market (TAM) - Expanded & Annualized

**Business Definition:** The absolute universe of potential customers. To transition from a raw survey sample to real-world national demographics, this calculation leverages the DANE Expansion Factor (`FEX_C18`). This weight variable projects our sampled individuals to the actual scale of the Colombian population aged 18 to 35.

**Technical Execution:** A `SUM` aggregation is applied to the `FEX_C18` variable instead of a standard row count. Furthermore, because the ingestion pipeline consolidated a 12-month longitudinal series (January - December 2025), a temporal duplication effect occurs. To correct this and determine the true baseline market size, the aggregated sum is annualized (divided by 12) to extract the average national population for the fiscal year.

In [3]:
# ==============================================================================
# DATA RETRIEVAL 1: TOTAL ADDRESSABLE MARKET (TAM)
# ==============================================================================
from google.cloud import bigquery
import pandas as pd

# Initialize the BigQuery client
client = bigquery.Client(project='avani-market-research')

# Validated SQL query correcting for longitudinal duplication (12 months)
query_tam = """
SELECT
    ROUND(SUM(FEX_C18) / 12, 0) AS Total_Population_TAM
FROM `avani-market-research.GEIH.stg_caracteristicas_generales`
WHERE CAST(P6040 AS INT64) BETWEEN 18 AND 35
"""

# Execute the query and store the results in a Pandas DataFrame
df_tam = client.query(query_tam).to_dataframe()

# Display the market intelligence results
print("TAM Results Retrieved Successfully (Annualized Average):")
display(df_tam)

TAM Results Retrieved Successfully (Annualized Average):


,Total_Population_TAM
0,14306524.0


### 2. Serviceable Available Market (SAM) - Income & Digital Readiness Filtered

**Market Research Definition:** The segment of the TAM that possesses both the purchasing power and the digital readiness to adopt a tech-enabled activewear and wellness subscription. Based on macroeconomic and fintech data, a minimum monthly labor income floor of **1,500,000 COP** was established. This threshold validates two critical market conditions:

1. **Liquidity for Wellness Expenditure:** Targeting this income floor isolates young adults with formal employment. This demographic demonstrates a healthy wallet share capacity for the activewear category, capable of sustaining recurring discretionary spending despite macroeconomic fluctuations (e.g., projected 5% inflation).
2. **Digital Banking Penetration:** Market research indicates that this formal income bracket correlates with high adoption of digital financial services (low-value deposits and digital wallets like Nequi/Daviplata, covering ~79% of digital transactions). This ensures the target population is technologically equipped to participate in recurring digital payment models without friction.

Data Sources:
* *Sectorial/RADDAR (2025) & Colombia Fintech (2024) Reports.*

**Technical Execution:** An `INNER JOIN` operation bridges the demographic and employment datasets (`stg_caracteristicas_generales` and `stg_ocupados`). The pipeline filters for the target age group (18-35) and strictly evaluates the `INGLABO` (Labor Income) variable against the >= 1.5M COP threshold, applying the annualized expansion factor (`FEX_C18`) to project the true available population.

In [4]:
# ==============================================================================
# DATA RETRIEVAL 2: SERVICEABLE AVAILABLE MARKET (SAM)
# ==============================================================================

# Validated SQL query extracting the target audience with viable purchasing power
query_sam = """
SELECT
    ROUND(SUM(c.FEX_C18) / 12, 0) AS Population_SAM
FROM `avani-market-research.GEIH.stg_caracteristicas_generales` c
INNER JOIN `avani-market-research.GEIH.stg_ocupados` o
    ON c.DIRECTORIO = o.DIRECTORIO
    AND c.SECUENCIA_P = o.SECUENCIA_P
    AND c.ORDEN = o.ORDEN
WHERE CAST(c.P6040 AS INT64) BETWEEN 18 AND 35
  AND o.INGLABO IS NOT NULL
  AND CAST(o.INGLABO AS FLOAT64) >= 1500000
"""

# Execute the query and store the results
df_sam = client.query(query_sam).to_dataframe()

# Display the market intelligence results
print("SAM Results Retrieved Successfully (Threshold >= 1.5M COP):")
display(df_sam)

SAM Results Retrieved Successfully (Threshold >= 1.5M COP):


,Population_SAM
0,3898725.0


### 3. Market Funnel & Financial Feasibility (SOM Formulation)
**Objective:** Translate the macro-economic population (TAM) into a strictly qualified Serviceable Obtainable Market (SOM) to project baseline revenue. This phase applies sequential strategic filters to the base population: a minimum income threshold, a 40% wellness/fitness engagement rate, and a 79% digital transaction readiness. Finally, assuming a highly conservative 1% market capture rate at a $35,000 COP monthly ticket, we model the Monthly Recurring Revenue (MRR) and the net Annual Recurring Revenue (ARR) factoring in estimated user churn.

In [5]:
# ==============================================================================
# PHASE II: FINANCIAL MODELING, FUNNEL CONSOLIDATION & PERSISTENCE
# ==============================================================================
from google.cloud import bigquery
import pandas as pd

print("Initiating Funnel Consolidation and BigQuery Sync...")

# 1. Business & Behavioral Assumptions
ACTIVE_POPULATION_RATE = 0.40     # 40% Physical activity / wellness interest rate
DIGITAL_READINESS_RATE = 0.79     # 79% Digital transaction adoption
TARGET_CAPTURE_RATE = 0.010       # 1.0% Penetration of the qualified market
MONTHLY_MEMBERSHIP_COP = 35000    # Estimated monthly subscription fee
ANNUAL_CHURN_RATE = 0.35          # 35% estimated annual user churn

PROJECT_ID = 'avani-market-research'
DATASET_ID = 'GEIH'
TABLE_NAME = 'avani_strategic_metrics_2025'
TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"

try:
    # 2. Extract Base Metrics (from previous SQL queries)
    total_tam_population = df_tam.iloc[0, 0]
    base_sam = df_sam['Population_SAM'].iloc[0]

    # 3. Apply Strategic Filters (Funnel Math)
    wellness_sam = int(base_sam * ACTIVE_POPULATION_RATE)
    digital_sam = int(wellness_sam * DIGITAL_READINESS_RATE)
    total_subscribers = int(digital_sam * TARGET_CAPTURE_RATE)

    # 4. Financial Modeling
    mrr_cop = total_subscribers * MONTHLY_MEMBERSHIP_COP
    gross_arr_cop = mrr_cop * 12
    net_arr_cop = gross_arr_cop * (1 - ANNUAL_CHURN_RATE)

    # 5. Create Master Funnel DataFrame
    master_data = {
        'step_order': [1, 2, 3, 4, 5, 6, 7],
        'metric_name': [
            'TAM: Total Population (18-35)',
            'Base SAM: Income Filtered',
            'Wellness SAM (40% Active)',
            'Qualified Digital SAM (79%)',
            'SOM: Target Subscribers (1%)',
            'Monthly Recurring Revenue (MRR)',
            'Net Annual Revenue (ARR)'
        ],
        'metric_value': [
            float(total_tam_population), float(base_sam), float(wellness_sam),
            float(digital_sam), float(total_subscribers), float(mrr_cop), float(net_arr_cop)
        ],
        'category': ['Volume', 'Volume', 'Behavioral', 'Tech', 'Commercial', 'Financial', 'Financial'],
        'unit': ['People', 'People', 'People', 'Users', 'Users', 'COP', 'COP']
    }
    df_master_funnel = pd.DataFrame(master_data)

    # 6. BigQuery Persistence (The Single Source of Truth)
    client = bigquery.Client(project=PROJECT_ID)
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    client.load_table_from_dataframe(df_master_funnel, TABLE_ID, job_config=job_config).result()

    print(f"\nSUCCESS: Master Funnel Table '{TABLE_NAME}' updated in BigQuery.")

    # 7. Professional Display Formatting for the Notebook
    df_display = df_master_funnel.copy()
    # Format numbers dynamically: Currency for COP, standard thousands for Users/People
    df_display['metric_value'] = df_display.apply(
        lambda x: f"${x['metric_value']:,.0f}" if x['unit'] == 'COP' else f"{x['metric_value']:,.0f}", axis=1
    )

    print("\n--- AVANI: STRATEGIC FUNNEL & FINANCIAL PROJECTIONS ---")
    display(df_display[['step_order', 'metric_name', 'metric_value', 'unit']])

except NameError as ne:
    print(f"ERROR: Missing base DataFrames. Ensure TAM and SAM cells were executed. Details: {ne}")
except Exception as e:
    print(f"ERROR during Funnel Consolidation: {e}")

Initiating Funnel Consolidation and BigQuery Sync...

SUCCESS: Master Funnel Table 'avani_strategic_metrics_2025' updated in BigQuery.

--- AVANI: STRATEGIC FUNNEL & FINANCIAL PROJECTIONS ---


,step_order,metric_name,metric_value,unit
0,1,TAM: Total Population (18-35),"14,306,524",People
1,2,Base SAM: Income Filtered,"3,898,725",People
2,3,Wellness SAM (40% Active),"1,559,490",People
3,4,Qualified Digital SAM (79%),"1,231,997",Users
4,5,SOM: Target Subscribers (1%),"12,319",Users
5,6,Monthly Recurring Revenue (MRR),"$431,165,000",COP
6,7,Net Annual Revenue (ARR),"$3,363,087,000",COP


### Phase II Conclusion: Strategic Viability & Revenue Projections
**Results:** The data-driven funnel successfully distilled a macro-population of 14.3M down to a highly qualified SOM of **12,319 target subscribers**.

The financial implications of this segmentation are highly favorable: at a conservative 1% market penetration, this user base unlocks an estimated **Monthly Recurring Revenue (MRR) of \$431.1M COP**. Factoring in an annual churn rate of 35%, the net **Annual Recurring Revenue (ARR) scales to \$3.36 Billion COP**. These metrics definitively validate the financial viability of the *Avani* subscription model, demonstrating high revenue potential and scalability even with minimal market capture.

To ensure data integrity for the Business Intelligence (BI) layer, this 7-tier hierarchical model has been synchronized with BigQuery, establishing our single source of truth for the upcoming geographic and demographic segmentation.

## **Phase III: Strategic Deep Dive - Geographic & Gender Distribution**
Beyond total volume, project viability depends on **localized precision**. By querying the `DPTO` (Department) and `P3271` (Gender) variables directly from the processed GEIH microdata, we can move from general assumptions to a data-driven "Go-to-Market" strategy.

### **1. Geographic Strategy (5% Relevance Threshold)**
To avoid operational fragmentation, we apply a strict **Relevance Threshold**: any department representing less than **5% of the SOM** is automatically clustered into a **"Rest of Colombia"** category. This ensures that our initial rollout and marketing efforts focus exclusively on the commercial hubs with the highest concentration of high-intent users.

### **2. Demographic Strategy (Gender Distribution)**
We extract the `P3271` variable to determine the precise Male/Female split within our validated SOM. Understanding this exact demographic composition is crucial for optimizing the Customer Acquisition Cost (CAC) through gender-specific digital ad creatives and targeted product merchandising.

In [6]:
# ==============================================================================
# PHASE III: MODULAR MARKET SEGMENTATION (REGION & GENDER)
# ==============================================================================
from google.cloud import bigquery
import pandas as pd
import pandas_gbq

# Clean visual configuration for Colab
pd.options.display.float_format = '{:.2f}'.format

print("Initiating Phase III: Generating modular tables...")

PROJECT_ID = 'avani-market-research'
DATASET_ID = 'GEIH'
client = bigquery.Client(project=PROJECT_ID)

try:
    # SOM result from Phase II
    final_som = total_subscribers

    # --------------------------------------------------------------------------
    # 1. REGIONAL DISTRIBUTION LOGIC
    # --------------------------------------------------------------------------
    query_geo = f"""
    SELECT
        CAST(c.DPTO AS INT64) as DPTO,
        SUM(c.FEX_C18) as Weight_Raw
    FROM `{PROJECT_ID}.{DATASET_ID}.stg_caracteristicas_generales` c
    INNER JOIN `{PROJECT_ID}.{DATASET_ID}.stg_ocupados` o
        ON c.DIRECTORIO = o.DIRECTORIO AND c.SECUENCIA_P = o.SECUENCIA_P AND c.ORDEN = o.ORDEN
    WHERE CAST(c.P6040 AS INT64) BETWEEN 18 AND 35
      AND CAST(o.INGLABO AS FLOAT64) >= 1500000
    GROUP BY 1
    """
    df_geo_raw = client.query(query_geo).to_dataframe()

    # Standard mapping for Colombia subdivisions
    dane_map = {
        5: 'Antioquia', 8: 'Atlantico', 11: 'Bogota', 13: 'Bolivar', 15: 'Boyaca',
        17: 'Caldas', 25: 'Cundinamarca', 47: 'Magdalena', 50: 'Meta',
        54: 'Norte de Santander', 66: 'Risaralda', 68: 'Santander',
        70: 'Sucre', 73: 'Tolima', 76: 'Valle del Cauca'
    }

    # Any department not in the core mapping goes to 'Rest of Colombia'
    df_geo_raw['Region'] = df_geo_raw['DPTO'].map(dane_map).fillna('Rest of Colombia')
    total_geo_pop = df_geo_raw['Weight_Raw'].sum()

    df_geo_grouped = df_geo_raw.groupby('Region')['Weight_Raw'].sum().reset_index()
    df_geo_grouped['Weight'] = df_geo_grouped['Weight_Raw'] / total_geo_pop

    # Filter: Greater than or equal to 5% AND not already 'Rest of Colombia'
    is_main_region = (df_geo_grouped['Weight'] >= 0.05) & (df_geo_grouped['Region'] != 'Rest of Colombia')

    main_regions = df_geo_grouped[is_main_region].copy()
    minor_weight = df_geo_grouped[~is_main_region]['Weight'].sum()

    df_rest = pd.DataFrame({'Region': ['Rest of Colombia'], 'Weight': [minor_weight]})
    df_geo_final = pd.concat([main_regions[['Region', 'Weight']], df_rest], ignore_index=True)

    # Financial calculations (Ticket: $35,000 COP)
    df_geo_final['Target_Users_SOM'] = (df_geo_final['Weight'] * final_som).astype(int)
    df_geo_final['Regional_MRR_COP'] = df_geo_final['Target_Users_SOM'] * 35000

    # --------------------------------------------------------------------------
    # 2. GENDER DISTRIBUTION LOGIC
    # --------------------------------------------------------------------------
    query_gender = f"""
    SELECT
        CAST(c.P3271 AS INT64) as Gender_Code,
        SUM(c.FEX_C18) as Weight_Raw
    FROM `{PROJECT_ID}.{DATASET_ID}.stg_caracteristicas_generales` c
    INNER JOIN `{PROJECT_ID}.{DATASET_ID}.stg_ocupados` o
        ON c.DIRECTORIO = o.DIRECTORIO AND c.SECUENCIA_P = o.SECUENCIA_P AND c.ORDEN = o.ORDEN
    WHERE CAST(c.P6040 AS INT64) BETWEEN 18 AND 35
      AND CAST(o.INGLABO AS FLOAT64) >= 1500000
    GROUP BY 1
    """
    df_gender_raw = client.query(query_gender).to_dataframe()
    df_gender_raw['Gender'] = df_gender_raw['Gender_Code'].map({1: 'Male', 2: 'Female'}).fillna('Unknown')
    total_gender_pop = df_gender_raw['Weight_Raw'].sum()

    df_gender_final = df_gender_raw.groupby('Gender')['Weight_Raw'].sum().reset_index()
    df_gender_final['Weight'] = df_gender_final['Weight_Raw'] / total_gender_pop
    df_gender_final['Target_Users_SOM'] = (df_gender_final['Weight'] * final_som).astype(int)
    df_gender_final['Gender_MRR_COP'] = df_gender_final['Target_Users_SOM'] * 35000
    df_gender_final = df_gender_final[['Gender', 'Weight', 'Target_Users_SOM', 'Gender_MRR_COP']]

    # --------------------------------------------------------------------------
    # 3. BIGQUERY SYNCHRONIZATION AND VISUALIZATION
    # --------------------------------------------------------------------------
    pandas_gbq.to_gbq(df_geo_final, f'{DATASET_ID}.regional_distribution_2025', project_id=PROJECT_ID, if_exists='replace')
    pandas_gbq.to_gbq(df_gender_final, f'{DATASET_ID}.demographic_gender_2025', project_id=PROJECT_ID, if_exists='replace')

    print("\nSUCCESS: Modular tables synchronized in BigQuery.")

    print("\n" + "="*50)
    print("--- GEOGRAPHIC DISTRIBUTION (SOM) ---")
    display(df_geo_final.sort_values(by='Weight', ascending=False))

    print("\n" + "="*50)
    print("--- GENDER DISTRIBUTION (SOM) ---")
    display(df_gender_final)

except Exception as e:
    print(f"ERROR: Phase III failed: {e}")

Initiating Phase III: Generating modular tables...


100%|██████████| 1/1 [00:00<00:00, 9467.95it/s]


SUCCESS: Modular tables synchronized in BigQuery.

--- GEOGRAPHIC DISTRIBUTION (SOM) ---


,Region,Weight,Target_Users_SOM,Regional_MRR_COP
5,Rest of Colombia,0.30,3717,130095000
2,Bogota,0.28,3493,122255000
0,Antioquia,0.20,2403,84105000
4,Valle del Cauca,0.09,1084,37940000
3,Cundinamarca,0.08,1002,35070000
1,Atlantico,0.05,618,21630000



--- GENDER DISTRIBUTION (SOM) ---


,Gender,Weight,Target_Users_SOM,Gender_MRR_COP
0,Female,0.42,5124,179340000
1,Male,0.58,7194,251790000


### Phase III Conclusion: Strategic Market Concentration & Rollout Feasibility
**Results:** The modular segmentation of the 12,319-user SOM reveals highly actionable concentration patterns that dictate a precise go-to-market strategy.

Geographically, the data supports a highly targeted urban rollout. **Bogotá (28.3%)** and **Antioquia (20.2%)** emerge as the undisputed primary commercial hubs, jointly commanding nearly half of the target market. By concentrating initial user acquisition budgets exclusively in these two regions before scaling to secondary hubs (Valle del Cauca, Cundinamarca), the business can aggressively optimize its Customer Acquisition Cost (CAC) and accelerate the path to profitability.

Demographically, the analysis reveals a significant **male-leaning target audience (58.7% Male vs. 42.5% Female)**. This insight is critical for the marketing team, establishing a data-driven foundation for visual identity, product messaging, and ad copy. It indicates that campaigns must break away from traditional female-centric wellness aesthetics and deploy strong male-focused creatives to capture the largest available market share.

By mapping the projected **\$431.1M COP Monthly Recurring Revenue (MRR)** directly to these specific regions and gender cohorts, the project successfully transitions from theoretical market sizing to a concrete, data-backed execution plan.

## **Phase IV: Data Architecture & Enterprise Export (BI Resource Layer)**

**Objective:** Transition from exploratory analysis to enterprise data warehousing. This phase serves as the final ETL pipeline, where refined market data is structured into dedicated resource tables within BigQuery.

These tables are specifically architected for **Power BI consumption**, ensuring that all financial metrics—including the **\$431.1M COP MRR** and **\$3.36B COP Net ARR**—are calculated upstream to maintain a "Single Source of Truth." By decoupling data processing from visualization, we optimize report performance and ensure data integrity across the executive dashboard.

In [7]:
# ==============================================================================
# PHASE IV: ENTERPRISE DATA EXPORT (BIGQUERY RESOURCE TABLES)
# ==============================================================================
from google.cloud import bigquery
import pandas as pd
import pandas_gbq

print("Starting Enterprise Data Export Pipeline...")

PROJECT_ID = 'avani-market-research' # Update with your project ID
DATASET_ID = 'GEIH'
client = bigquery.Client(project=PROJECT_ID)

# Financial Constants
TICKET_COP = 35000
CHURN_RATE = 0.35
ANNUAL_RETENTION = 1 - CHURN_RATE

try:
    # 1. FUNNEL RESOURCE TABLE
    # Consolidating the 5-step funnel for the Power BI Funnel Visual
    funnel_data = {
        'Step_Order': [1, 2, 3, 4, 5],
        'Stage': ['TAM: Total Population', 'Base SAM: Income Filtered', 'Wellness SAM (40%)', 'Digital SAM (79%)', 'Target SOM (1%)'],
        'User_Count': [total_tam_population, base_sam, wellness_sam, digital_sam, total_subscribers]
    }
    df_funnel_export = pd.DataFrame(funnel_data)

    # 2. FINANCIAL & REGIONAL RESOURCE TABLE
    # Using df_geo_final from Phase III to create the regional source
    df_regional_export = df_geo_final.copy()
    df_regional_export['Annual_Net_Revenue_ARR'] = df_regional_export['Regional_MRR_COP'] * 12 * ANNUAL_RETENTION

    # 3. DEMOGRAPHIC RESOURCE TABLE
    # Using df_gender_final from Phase III
    df_gender_export = df_gender_final.copy()
    df_gender_export['Gender_ARR_COP'] = df_gender_export['Gender_MRR_COP'] * 12 * ANNUAL_RETENTION

    # ==========================================================================
    # 3.5. CROSS-FILTER RESOURCE TABLE (NEW FOR POWER BI INTERACTIVITY)
    # Architecturally aligned with Phase III: Extracting Base SAM natively via SQL,
    # applying the 5% Relevance Threshold, and scaling exactly to Target SOM.
    # ==========================================================================
    query_cross = f"""
    SELECT
        CAST(c.DPTO AS INT64) as DPTO,
        CAST(c.P3271 AS INT64) as Gender_Code,
        SUM(c.FEX_C18) as Weight_Raw
    FROM `{PROJECT_ID}.{DATASET_ID}.stg_caracteristicas_generales` c
    INNER JOIN `{PROJECT_ID}.{DATASET_ID}.stg_ocupados` o
        ON c.DIRECTORIO = o.DIRECTORIO AND c.SECUENCIA_P = o.SECUENCIA_P AND c.ORDEN = o.ORDEN
    WHERE CAST(c.P6040 AS INT64) BETWEEN 18 AND 35
      AND CAST(o.INGLABO AS FLOAT64) >= 1500000
    GROUP BY 1, 2
    """
    df_cross_raw = client.query(query_cross).to_dataframe()

    # Apply DANE Mappings
    dane_map = {
        5: 'Antioquia', 8: 'Atlantico', 11: 'Bogota', 13: 'Bolivar', 15: 'Boyaca',
        17: 'Caldas', 25: 'Cundinamarca', 47: 'Magdalena', 50: 'Meta',
        54: 'Norte de Santander', 66: 'Risaralda', 68: 'Santander',
        70: 'Sucre', 73: 'Tolima', 76: 'Valle del Cauca'
    }
    df_cross_raw['Region'] = df_cross_raw['DPTO'].map(dane_map).fillna('Rest of Colombia')
    df_cross_raw['Gender'] = df_cross_raw['Gender_Code'].map({1: 'Male', 2: 'Female'}).fillna('Unknown')

    # Apply the exact 5% relevance threshold established in Phase III dynamically
    valid_regions = df_regional_export['Region'].tolist()
    df_cross_raw['Region'] = df_cross_raw['Region'].apply(lambda x: x if x in valid_regions else 'Rest of Colombia')

    # Group by Region and Gender, calculate structural weights
    df_cross_grouped = df_cross_raw.groupby(['Region', 'Gender'])['Weight_Raw'].sum().reset_index()
    total_cross_pop = df_cross_grouped['Weight_Raw'].sum()
    df_cross_grouped['Weight'] = df_cross_grouped['Weight_Raw'] / total_cross_pop

    # Apply Funnel Math to perfectly match the 12,319 Target SOM
    df_cross_grouped['Target_Users_SOM'] = (df_cross_grouped['Weight'] * total_subscribers).astype(int)

    # Isolate the exact columns needed for the BI Bridge
    df_gender_by_region_export = df_cross_grouped[['Region', 'Gender', 'Target_Users_SOM']]

    # 4. SILENT UPLOAD TO BIGQUERY
    # Replacing existing tables with optimized BI versions
    pandas_gbq.to_gbq(df_funnel_export, f'{DATASET_ID}.bi_funnel_stats', project_id=PROJECT_ID, if_exists='replace')
    pandas_gbq.to_gbq(df_regional_export, f'{DATASET_ID}.bi_regional_projections', project_id=PROJECT_ID, if_exists='replace')
    pandas_gbq.to_gbq(df_gender_export, f'{DATASET_ID}.bi_gender_demographics', project_id=PROJECT_ID, if_exists='replace')

    # Injecting the new cross-filter table
    pandas_gbq.to_gbq(df_gender_by_region_export, f'{DATASET_ID}.bi_gender_by_region', project_id=PROJECT_ID, if_exists='replace')

    print("\n" + "="*50)
    print("SUCCESS: ALL BI RESOURCE TABLES UPLOADED TO BIGQUERY.")
    print(f"Destination Dataset: {DATASET_ID}")
    print("Tables Ready for Power BI Connection:")
    print(" - bi_funnel_stats")
    print(" - bi_regional_projections")
    print(" - bi_gender_demographics")
    print(" - bi_gender_by_region (New Interactivity Bridge)")
    print("="*50)

except Exception as e:
    print(f"CRITICAL ERROR in Export Pipeline: {e}")

Starting Enterprise Data Export Pipeline...


100%|██████████| 1/1 [00:00<00:00, 10782.27it/s]


SUCCESS: ALL BI RESOURCE TABLES UPLOADED TO BIGQUERY.
Destination Dataset: GEIH
Tables Ready for Power BI Connection:
 - bi_funnel_stats
 - bi_regional_projections
 - bi_gender_demographics
 - bi_gender_by_region (New Interactivity Bridge)


### **Phase IV Conclusion: Deployment & Final Business Validation**
**CRISP-DM Stage 6: Deployment**

The automated ETL process successfully transitioned the raw macro-economic data into a finalized, enterprise-grade architecture. By synchronizing the strategic metrics—including the **12,319 Target SOM**, the projected **\$431.1M COP Monthly Recurring Revenue (MRR)**, and the **\$3.36B COP Net ARR**—into optimized BigQuery resource tables, the project establishes a "Single Source of Truth" for executive decision-making.

**Final Project Synthesis:**
This analysis proves that despite starting with a raw national population of over 14.3 million, applying rigorous, data-driven filters (income liquidity, digital readiness, and behavioral interest) reveals a highly concentrated and financially viable core market.

The strategic recommendation for the brand launch is unequivocally clear:
1. **Geographic Focus:** Initial Customer Acquisition efforts and logistics must be strictly isolated to **Bogotá (28.3%)** and **Antioquia (20.2%)** to optimize the CAC (Customer Acquisition Cost) before scaling nationally.
2. **Demographic Targeting:** Brand positioning and marketing creatives must pivot to capture the dominant **male-leaning audience (58.7%)**, moving away from traditional female-centric wellness aesthetics.

By decoupling the heavy data processing (Google Colab / BigQuery) from the visualization layer, this deployment guarantees a highly performant, interactive Power BI dashboard ready for stakeholder consumption.

In [8]:
from IPython.display import display, HTML

POWER_BI_URL = "https://app.powerbi.com/view?r=eyJrIjoiMjYyM2JlY2EtZGRjOC00ZWNiLWE1ZmItNDhiOWFlMTZhMmE3IiwidCI6ImJlZjU3Y2ZhLWY1MzItNDU4Ny05ZTY4LWJhNDEyYTU1OTY0YiIsImMiOjR9"

display(HTML(f"""
    <div style="border: 1px solid #ddd; border-radius: 8px; overflow: hidden;">
        <iframe
            title="Market Feasibility Dashboard"
            width="100%"
            height="600"
            src="{POWER_BI_URL}"
            frameborder="0"
            allowFullScreen="true">
        </iframe>
    </div>
"""))

### **References & Data Sources**

This project was built using public macroeconomic microdata and industry-standard analytical frameworks. All data processing and transformations are fully documented within this notebook.

**1. Primary Data Sources & Industry Reports:**
* **DANE (Departamento Administrativo Nacional de Estadística).** (n.d.). *Empleo y desempleo - Históricos Gran Encuesta Integrada de Hogares (GEIH)*. Microdata utilized for demographic structuring, formal labor income mapping, and geographic distribution across Colombia. Retrieved April 22, 2026, from [DANE Official Portal](https://www.dane.gov.co/index.php/estadisticas-por-tema/mercado-laboral/empleo-y-desempleo/geih-historicos)
* **Cámara Colombiana de Comercio Electrónico [CCCE].** (2025). *Informe comercio electrónico minorista 2025*. Used for digital consumer behavior parameters. [CCCE Official Portal](https://www.ccce.org.co/)
* **Colombia Fintech.** (2025). *Fintech snapshot: Pagos digitales*. Used to validate digital banking readiness metrics. [Colombia Fintech Portal](https://www.colombiafintech.co/)
* **Sectorial & RADDAR.** (2025). *Informes Observatorio Moda (Textiles y Confecciones: Enero, Abril, Septiembre, Noviembre, Diciembre 2025)*. Laboratorio de Conocimiento Inexmoda. Utilized for industry-specific market behavior and wellness sector growth trends. [Inexmoda Official Portal](https://www.inexmoda.org.co/)

**2. Methodological Framework:**
* **CRISP-DM:** *Cross-Industry Standard Process for Data Mining*. Applied to structure the project lifecycle from Business Understanding to final Business Intelligence Deployment.

**3. Technology Stack:**
* **Data Warehouse & SQL Engine:** Google Cloud BigQuery.
* **Data Pipeline & Processing:** Python (`pandas`, `google-cloud-bigquery`).
* **Business Intelligence & Visualization:** Microsoft Power BI.

*Disclaimer: The strategic business concept and brand presented in this study act as a conceptual framework used exclusively to demonstrate data engineering capabilities, market sizing methodologies, and financial modeling. All underlying population data remains strictly bound to DANE's official statistical anonymization protocols.*